# RSNA Knee Abnormality Detection - 2.5D Multi-Plane Submission
Self-contained offline inference kernel qualifying for the $17,000 Model Efficiency Track.

In [ ]:
"""RSNA Knee Abnormality Detection - Standalone Offline Inference Script.

Auto-generated by SubmissionExporter for air-gapped Kaggle competition submission.
Executes high-throughput DICOM slab extraction, multi-plane attention fusion,
and generates submission.csv qualifying for the $17,000 Model Efficiency Track.
"""

import os
import sys
import time
import glob
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

# Competition Target Column Order
TARGET_COLS = ['acl', 'pcl', 'mcl', 'lcl', 'medial_meniscus', 'lateral_meniscus', 'patellar_tendon', 'quadriceps_tendon', 'cartilage_femur', 'cartilage_tibia', 'cartilage_patella', 'joint_effusion']

DATA_DIR = Path("/kaggle/input/rsna-knee-abnormality-detection")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

WEIGHTS_DIR = Path("/kaggle/input/rsna-knee-weights")
if not WEIGHTS_DIR.exists():
    WEIGHTS_DIR = Path("checkpoints/joint")

OUTPUT_CSV = Path("submission.csv")


# --- 1. FAST DICOM READING & SLAB EXTRACTION ---

def read_dicom_slice(path: Union[str, Path]) -> Tuple[np.ndarray, Optional[List[float]], float]:
    """Read a single DICOM slice with pydicom and extract orientation cosines."""
    dcm = pydicom.dcmread(str(path), stop_before_pixels=False)
    arr = dcm.pixel_array.astype(np.float32)

    slope = float(getattr(dcm, "RescaleSlope", 1.0))
    intercept = float(getattr(dcm, "RescaleIntercept", 0.0))
    arr = arr * slope + intercept

    photometric = getattr(dcm, "PhotometricInterpretation", "MONOCHROME2")
    if photometric == "MONOCHROME1":
        arr = np.max(arr) - arr

    iop = getattr(dcm, "ImageOrientationPatient", None)
    iop_list = [float(x) for x in iop] if iop else None

    ipp = getattr(dcm, "ImagePositionPatient", None)
    z_pos = float(ipp[2]) if ipp and len(ipp) >= 3 else 0.0

    return arr, iop_list, z_pos


def classify_plane(iop: Optional[List[float]]) -> str:
    """Classify series into sagittal, coronal, or axial via normal vector math."""
    if not iop or len(iop) < 6:
        return "sagittal"
    r = np.array(iop[0:3], dtype=np.float32)
    c = np.array(iop[3:6], dtype=np.float32)
    n = np.cross(r, c)
    norm = np.linalg.norm(n)
    if norm > 1e-6:
        n = n / norm
    nx, ny, nz = abs(n[0]), abs(n[1]), abs(n[2])
    if nx >= ny and nx >= nz:
        return "sagittal"
    elif ny >= nx and ny >= nz:
        return "coronal"
    else:
        return "axial"


def extract_25d_slabs(volume: np.ndarray, num_slabs: int = 4, target_shape: Tuple[int, int] = (256, 256)) -> torch.Tensor:
    """Extract contiguous 3-channel 2.5D slabs with linear windowing."""
    n_slices, h, w = volume.shape
    # Linear windowing
    p1, p99 = np.percentile(volume, (1.0, 99.0))
    if p99 > p1:
        norm_vol = np.clip((volume - p1) / (p99 - p1), 0.0, 1.0)
    else:
        norm_vol = np.zeros_like(volume)

    centers = np.linspace(0, max(0, n_slices - 1), num=num_slabs).astype(int)
    slabs = []
    for c in centers:
        i_prev = max(0, c - 1)
        i_curr = c
        i_next = min(n_slices - 1, c + 1)
        slab = np.stack([norm_vol[i_prev], norm_vol[i_curr], norm_vol[i_next]], axis=0)  # (3, H, W)
        # Fast center crop/resize
        slab_t = torch.from_numpy(slab).unsqueeze(0).float()
        if (h, w) != target_shape:
            slab_t = F.interpolate(slab_t, size=target_shape, mode="bilinear", align_corners=False)
        slabs.append(slab_t.squeeze(0))

    return torch.stack(slabs, dim=0)  # (S, 3, H, W)


# --- 2. MULTI-PLANE ATTENTION VISION MODEL ---

class GatedAttentionPool(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int = 128):
        super().__init__()
        self.attn_v = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.Tanh())
        self.attn_u = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.Sigmoid())
        self.attn_w = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # x: (B, S, D)
        v = self.attn_v(x)
        u = self.attn_u(x)
        weights = F.softmax(self.attn_w(v * u), dim=1)  # (B, S, 1)
        pooled = (x * weights).sum(dim=1)  # (B, D)
        return pooled, weights


class MultiPlaneFusion(nn.Module):
    def __init__(self, feature_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.planes = ["sagittal", "coronal", "axial"]
        self.plane_embeddings = nn.Parameter(torch.randn(3, feature_dim) * 0.02)
        self.missing_token = nn.Parameter(torch.zeros(1, 1, feature_dim))
        self.mha = nn.MultiheadAttention(feature_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(feature_dim)
        self.norm2 = nn.LayerNorm(feature_dim)
        self.proj = nn.Sequential(
            nn.Linear(feature_dim * 3, feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, feature_dim),
        )

    def forward(self, plane_feats: Dict[str, torch.Tensor]) -> torch.Tensor:
        sample = next(iter(plane_feats.values()))
        B, D = sample.size(0), sample.size(1)
        device, dtype = sample.device, sample.dtype

        tokens_list = []
        for p_name in self.planes:
            if p_name in plane_feats and plane_feats[p_name] is not None:
                tokens_list.append(plane_feats[p_name])
            else:
                tokens_list.append(self.missing_token.expand(B, 1, -1).squeeze(1).to(device, dtype))

        tokens = torch.stack(tokens_list, dim=1) + self.plane_embeddings.unsqueeze(0).to(device, dtype)
        attn_out, _ = self.mha(tokens, tokens, tokens)
        tokens = self.norm1(tokens + attn_out)
        fused = self.norm2(self.proj(tokens.flatten(start_dim=1)))
        return fused


class MultiPlaneKneeModel(nn.Module):
    def __init__(self, feature_dim: int = 512, num_classes: int = len(TARGET_COLS)):
        super().__init__()
        # Convolutional feature extractor (Lightweight ResNet block)
        self.conv = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, feature_dim, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.pools = nn.ModuleDict({p: GatedAttentionPool(feature_dim) for p in ["sagittal", "coronal", "axial"]})
        self.fusion = MultiPlaneFusion(feature_dim, num_heads=4)
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, plane_dict: Dict[str, torch.Tensor]) -> torch.Tensor:
        plane_embeddings = {}
        for p_name, x in plane_dict.items():
            # x: (B, S, 3, H, W)
            B, S, C, H, W = x.shape
            folded = x.view(B * S, C, H, W)
            feats = self.conv(folded).view(B, S, -1)
            pooled, _ = self.pools[p_name](feats)
            plane_embeddings[p_name] = pooled

        fused = self.fusion(plane_embeddings)
        return self.classifier(fused)

    @torch.no_grad()
    def predict_proba(self, plane_dict: Dict[str, torch.Tensor]) -> np.ndarray:
        logits = self.forward(plane_dict)
        return torch.sigmoid(logits).cpu().numpy()


# --- 3. TEST INFERENCE LOOP ---

def run_inference():
    print("Beginning RSNA Knee multi-plane inference...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Inference device: {device}")

    model = MultiPlaneKneeModel(num_classes=len(TARGET_COLS)).to(device)
    model.eval()

    # Load weights if available
    weights_path = WEIGHTS_DIR / "best_multiplane_model.pt"
    if weights_path.exists():
        print(f"Loading checkpoint weights from {weights_path}...")
        ckpt = torch.load(weights_path, map_location=device)
        state_dict = ckpt.get("model_state_dict", ckpt)
        try:
            model.load_state_dict(state_dict, strict=False)
            print("Successfully loaded model weights.")
        except Exception as e:
            print(f"Warning loading weights: {e}")

    # Load test studies
    test_csv = DATA_DIR / "test.csv"
    if test_csv.exists():
        test_df = pd.read_csv(test_csv)
    else:
        # Fallback if testing locally
        sample_sub = DATA_DIR / "sample_submission.csv"
        if sample_sub.exists():
            test_df = pd.read_csv(sample_sub)
        else:
            test_df = pd.DataFrame({"StudyInstanceUID": ["test_study_001"]})

    print(f"Total test studies: {len(test_df)}")
    test_series_dir = DATA_DIR / "test_series"

    results = []
    start_time = time.time()

    with torch.inference_mode():
        for idx, row in test_df.iterrows():
            study_id = row["StudyInstanceUID"]
            study_path = test_series_dir / study_id

            plane_tensors = {}
            if study_path.exists():
                series_folders = [f for f in study_path.iterdir() if f.is_dir()]
                for s_dir in series_folders:
                    dcm_files = sorted(list(s_dir.glob("*.dcm")))
                    if not dcm_files:
                        continue
                    slices, z_positions, plane_name = [], [], None
                    for d_file in dcm_files[:30]:  # Limit max slices for speed
                        try:
                            arr, iop, z = read_dicom_slice(d_file)
                            slices.append(arr)
                            z_positions.append(z)
                            if plane_name is None:
                                plane_name = classify_plane(iop)
                        except Exception:
                            continue

                    if slices and plane_name:
                        # Sort by z-position
                        order = np.argsort(z_positions)
                        vol = np.stack([slices[i] for i in order], axis=0)
                        slab_tensor = extract_25d_slabs(vol, num_slabs=4, target_shape=(256, 256))
                        plane_tensors[plane_name] = slab_tensor.unsqueeze(0).to(device)

            if not plane_tensors:
                # Corrupted or missing: fallback to prior
                probs = np.full((1, len(TARGET_COLS)), 0.10, dtype=np.float32)
            else:
                # Predict probabilities
                with torch.autocast(device_type=device.type, enabled=True):
                    probs = model.predict_proba(plane_tensors)

            res_entry = {"StudyInstanceUID": study_id}
            for c_idx, c_name in enumerate(TARGET_COLS):
                res_entry[c_name] = float(np.clip(probs[0, c_idx], 0.0, 1.0))
            results.append(res_entry)

    elapsed = time.time() - start_time
    print(f"Inference complete in {elapsed:.2f}s ({elapsed / max(1, len(test_df)):.3f}s/study).")

    sub_df = pd.DataFrame(results)
    sub_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved submission to {OUTPUT_CSV} with shape {sub_df.shape}.")


if __name__ == "__main__":
    run_inference()
